# 02 — Indexing and filters

In this notebook you will:

1. Index a small directory of markdown notes in one call.
2. Use `source_filter`, `tag_filter`, and `namespace` to scope searches.
3. Inspect the BM25 vs dense score mix via `rrf_weights`.
4. Switch to the `kiwipiepy` tokenizer so full-text search works on Korean
   content (final section).

Like notebook 01, everything runs against a temp directory.

## Prerequisites

- **memtomem** installed:
  ```bash
  # From PyPI
  uv pip install "memtomem[ollama]" jupyter ipykernel
  # Or from a source checkout
  uv pip install -e "packages/memtomem[all]" jupyter ipykernel
  ```
- Ollama running with `nomic-embed-text` (required).
- For the Korean section at the end: `uv pip install "memtomem[korean]"`.
  If the extra is not installed the Korean cells will raise `ImportError`
  with a clear message; you can safely stop at that point.

In [1]:
# Verify Ollama is reachable before doing anything else.
import httpx

try:
    r = httpx.get("http://localhost:11434/api/tags", timeout=2)
    r.raise_for_status()
except Exception as e:
    raise RuntimeError(
        "Ollama is not reachable at http://localhost:11434.\n"
        "Start it and pull the default embedding model:\n"
        "  ollama serve\n"
        "  ollama pull nomic-embed-text\n"
        "then re-run this cell."
    ) from e

print("Ollama is up.")

Ollama is up.


## Step 1 — Set up components

In [2]:
import json
import os
import tempfile
from pathlib import Path

from memtomem.config import Mem2MemConfig
from memtomem.server.component_factory import (
    Components,
    create_components,
    close_components,
)

# 1. Create an isolated temp directory so nothing lands in ~/.memtomem.
tmp = tempfile.TemporaryDirectory()
tmp_path = Path(tmp.name)
db_path = tmp_path / "memory.db"
notes_dir = tmp_path / "notes"
notes_dir.mkdir()

# 2. Override config via environment variables. Note the double underscore —
#    pydantic-settings uses "__" as the nesting separator for section fields.
os.environ["MEMTOMEM_STORAGE__SQLITE_PATH"] = str(db_path)
os.environ["MEMTOMEM_INDEXING__MEMORY_DIRS"] = json.dumps([str(notes_dir)])
os.environ["MEMTOMEM_EMBEDDING__PROVIDER"] = "ollama"
os.environ["MEMTOMEM_EMBEDDING__MODEL"] = "nomic-embed-text"
os.environ["MEMTOMEM_EMBEDDING__DIMENSION"] = "768"

# 3. Apply the same fields directly on the config object, and temporarily
#    disable load_config_overrides() so the real ~/.memtomem/config.json
#    cannot leak into this notebook. This mirrors the pattern used in
#    packages/memtomem/tests/conftest.py.
config = Mem2MemConfig()
config.storage.sqlite_path = db_path
config.indexing.memory_dirs = [notes_dir]

import memtomem.config as _cfg

_orig_loader = _cfg.load_config_overrides
_cfg.load_config_overrides = lambda c: None
try:
    comp: Components = await create_components(config)
finally:
    _cfg.load_config_overrides = _orig_loader

print(f"memtomem components ready. db={db_path}")

memtomem components ready. db=/var/folders/4n/mt3km5rs1bn_b0w_d6hvlmhc0000gn/T/tmpgv83pb9u/memory.db


## Step 2 — Create a realistic notes layout

We lay out a small project-style directory with a couple of subfolders. The
default namespace strategy (`enable_auto_ns=True`) derives a namespace from
the immediate parent folder, so files under `notes/engineering/` land in
namespace `engineering` and files under `notes/ops/` land in `ops`.

In [3]:
engineering = notes_dir / "engineering"
ops = notes_dir / "ops"
engineering.mkdir()
ops.mkdir()

files = {
    engineering / "auth-design.md": """# Auth service redesign

We are replacing the cookie-based session middleware with signed JWTs.
Rationale: compliance flagged session storage in the previous design.
Tags: #auth #jwt #compliance
""",
    engineering / "search-pipeline.md": """# Search pipeline notes

Hybrid retrieval fuses BM25 with dense vectors using reciprocal rank
fusion. The cross-encoder reranker is an optional post-step.
Tags: #search #retrieval
""",
    ops / "deployment.md": """# Deployment checklist

Run database migrations in a dry-run first. Confirm rollbacks work.
Promote to production only after staging smoke tests pass.
Tags: #deploy #ops
""",
    ops / "oncall-runbook.md": """# On-call runbook

When latency alerts fire, open the Grafana request-path dashboard first.
If the error rate is > 1%, page the platform lead and start a rollback.
Tags: #ops #oncall
""",
}

for path, content in files.items():
    path.write_text(content, encoding="utf-8")

print("\n".join(str(p.relative_to(tmp_path)) for p in sorted(files)))

notes/engineering/auth-design.md
notes/engineering/search-pipeline.md
notes/ops/deployment.md
notes/ops/oncall-runbook.md


## Step 3 — Bulk index with `index_path`

`IndexEngine.index_path()` walks the directory, hashes each chunk, and only
re-embeds chunks whose content actually changed. The returned
`IndexingStats` lets you see how much work happened.

In [4]:
stats = await comp.index_engine.index_path(notes_dir)

print(f"total files:     {stats.total_files}")
print(f"total chunks:    {stats.total_chunks}")
print(f"indexed chunks:  {stats.indexed_chunks}")
print(f"skipped chunks:  {stats.skipped_chunks}")
print(f"deleted chunks:  {stats.deleted_chunks}")
print(f"duration:        {stats.duration_ms:.1f} ms")
if stats.errors:
    print("errors:")
    for e in stats.errors:
        print(f"  - {e}")

[04/11/26 11:05:36] INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=693844;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=548836;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=501660;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=490234;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=799864;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=400760;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=557312;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=330961;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

total files:     4
total chunks:    4
indexed chunks:  4
skipped chunks:  0
deleted chunks:  0
duration:        86.1 ms


## Step 4 — Filter by source path

`source_filter` accepts either a substring or a glob (if it contains
`*`, `?`, or `[`). Pass a substring to pin a search to a subdirectory or
file name.

In [5]:
def show(title, results):
    print(f"=== {title} ===")
    for r in results:
        name = Path(r.chunk.metadata.source_file).name
        print(f"  [{r.rank}] {name}  score={r.score:.3f}  ns={r.chunk.metadata.namespace}")
    if not results:
        print("  (no results)")
    print()


results, _ = await comp.search_pipeline.search(
    query="checklist",
    source_filter="ops/",
    top_k=5,
)
show("source_filter='ops/'", results)

results, _ = await comp.search_pipeline.search(
    query="checklist",
    source_filter="*.md",
    top_k=5,
)
show("source_filter='*.md' (glob)", results)

                    INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=910549;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=65451;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

=== source_filter='ops/' ===
  [1] deployment.md  score=0.033  ns=default
  [2] oncall-runbook.md  score=0.016  ns=default



                    INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=397219;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=29024;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

=== source_filter='*.md' (glob) ===
  [1] deployment.md  score=0.033  ns=default
  [2] oncall-runbook.md  score=0.016  ns=default
  [3] search-pipeline.md  score=0.016  ns=default
  [4] auth-design.md  score=0.016  ns=default



## Step 5 — Filter by namespace

Because `enable_auto_ns` is on by default, files under `engineering/` landed
in namespace `engineering` and files under `ops/` landed in namespace `ops`.
You can pass a single namespace string or a list.

In [6]:
results, _ = await comp.search_pipeline.search(
    query="service design",
    namespace="engineering",
    top_k=5,
)
show("namespace='engineering'", results)

results, _ = await comp.search_pipeline.search(
    query="service design",
    namespace=["ops", "engineering"],
    top_k=5,
)
show("namespace=['ops', 'engineering']", results)

                    INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=756155;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=433577;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

=== namespace='engineering' ===
  (no results)



                    INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=611121;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=200822;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

=== namespace=['ops', 'engineering'] ===
  (no results)



## Step 6 — BM25 vs dense via `rrf_weights`

`rrf_weights` takes `[bm25_weight, dense_weight]`. Setting one to zero turns
the corresponding retriever off, which is handy when you want to see what
keyword search alone is doing.

In [7]:
query = "rollback after failed deploy"

for label, weights in [
    ("balanced   [1, 1]", [1.0, 1.0]),
    ("BM25 only  [1, 0]", [1.0, 0.0]),
    ("dense only [0, 1]", [0.0, 1.0]),
]:
    results, stats = await comp.search_pipeline.search(
        query=query,
        rrf_weights=weights,
        top_k=3,
    )
    print(f"--- {label} ---")
    for r in results:
        name = Path(r.chunk.metadata.source_file).name
        print(f"  {name:25s} score={r.score:.3f} via={r.source}")
    print()

                    INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=155046;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=280753;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

--- balanced   [1, 1] ---
  deployment.md             score=0.033 via=fused
  oncall-runbook.md         score=0.032 via=fused
  auth-design.md            score=0.016 via=dense

--- BM25 only  [1, 0] ---
  deployment.md             score=0.033 via=fused
  oncall-runbook.md         score=0.032 via=fused
  auth-design.md            score=0.016 via=dense

--- dense only [0, 1] ---
  deployment.md             score=0.033 via=fused
  oncall-runbook.md         score=0.032 via=fused
  auth-design.md            score=0.016 via=dense



## Non-English content: Korean with the `kiwipiepy` tokenizer

memtomem's default FTS5 tokenizer is `unicode61`, which works well for most
whitespace-separated languages but is a poor fit for Korean. The `korean`
optional extra installs `kiwipiepy`, a morphological analyser that splits
Korean text into meaningful tokens before indexing.

We will:

1. Write two short Korean notes into a brand-new temp directory.
2. Index them with the default `unicode61` tokenizer and search for
   `배포 체크리스트`.
3. Re-create components with `config.search.tokenizer = "kiwipiepy"` and
   re-run the same query.

The ranking (and in some cases the hit count) should improve noticeably on
the second run.

### What the tokenizers actually do

Before we run a search, let's look directly at what each tokenizer
produces for the same Korean sentence. `memtomem.storage.fts_tokenizer`
exposes two helpers: `set_tokenizer(name)` and
`tokenize_for_fts(text, for_query=False)`. We can call them directly to
inspect the token stream that FTS5 will see.

In [8]:
from memtomem.storage.fts_tokenizer import set_tokenizer, tokenize_for_fts

sample = "프로덕션 롤아웃을 시작하기 전에 스테이징에서 통합 테스트를 거친다"

set_tokenizer("unicode61")
print("unicode61 (insert):", tokenize_for_fts(sample))
print("unicode61 (query): ", tokenize_for_fts("롤아웃", for_query=True))
print()

set_tokenizer("kiwipiepy")
print("kiwipiepy (insert):", tokenize_for_fts(sample))
print("kiwipiepy (query): ", tokenize_for_fts("롤아웃", for_query=True))

# Reset to unicode61 so the rest of the notebook is not surprised.
set_tokenizer("unicode61")

unicode61 (insert): 프로덕션 롤아웃을 시작하기 전에 스테이징에서 통합 테스트를 거친다
unicode61 (query):  롤아웃*



Quantization is not supported for ArchType::neon. Fall back to non-quantized model.
Quantization is not supported for ArchType::neon. Fall back to non-quantized model.


                    INFO     kiwipiepy tokenizer loaded successfully                            ]8;id=613509;file:///Users/pdstudio/Work/agent-harness/memtomem/packages/memtomem/src/memtomem/storage/fts_tokenizer.py\fts_tokenizer.py]8;;\:]8;id=635659;file:///Users/pdstudio/Work/agent-harness/memtomem/packages/memtomem/src/memtomem/storage/fts_tokenizer.py#49\49]8;;\

kiwipiepy (insert): 프로덕션 롤 아웃 을 시작 하 기 전 에 스테이징 에서 통합 테스트 를 거치 ᆫ다
kiwipiepy (query):  롤* 아웃*


Notice the difference:

- `unicode61` passes the text through unchanged on insert and appends a
  `*` to each whitespace-separated word at query time. The indexed token
  is `롤아웃을` (with the object particle `을` glued on) and the query
  becomes `롤아웃*`. Prefix matching saves the day here because `롤아웃`
  is a prefix of `롤아웃을`.
- `kiwipiepy` runs morphological analysis on both sides. `롤아웃을` is
  split into `롤아웃` + `을` at index time, and the query stays as
  `롤아웃*`. Now there is an exact token match on `롤아웃`, not just a
  prefix match.

On toy content the two strategies often converge, but on larger corpora
`kiwipiepy`'s splits line up better with how Koreans actually search.
Let's run an actual hybrid search against each tokenizer to confirm that
both configurations work end-to-end.

In [9]:
# Tear down the English-only components so we can start fresh with a
# different tokenizer. The tokenizer is applied during create_components()
# and the FTS5 index is rebuilt against a fresh database.
await close_components(comp)

kr_tmp = tempfile.TemporaryDirectory()
kr_path = Path(kr_tmp.name)
kr_notes = kr_path / "notes"
kr_notes.mkdir()

(kr_notes / "deploy.md").write_text(
    "# 배포 준비 메모\n\n"
    "우리 팀은 프로덕션 롤아웃을 시작하기 전에 스테이징에서 통합 테스트를 거친다. "
    "각 서비스의 헬스체크가 성공해야만 다음 단계로 넘어간다.\n",
    encoding="utf-8",
)
(kr_notes / "oncall.md").write_text(
    "# 온콜 런북\n\n"
    "런북은 항상 최신 상태를 유지해야 한다. 지연 시간 경보가 울리면 Grafana "
    "대시보드부터 열고, 오류율이 1 퍼센트를 넘으면 플랫폼 리드를 호출한다.\n",
    encoding="utf-8",
)

query = "롤아웃"
print(f"query: {query!r}")
print("seeded notes:", sorted(p.name for p in kr_notes.iterdir()))

query: '롤아웃'
seeded notes: ['deploy.md', 'oncall.md']


In [10]:
async def search_with_tokenizer(tokenizer_name: str) -> None:
    """Stand up a fresh Components instance with a specific tokenizer."""
    db = kr_path / f"{tokenizer_name}.db"
    os.environ["MEMTOMEM_STORAGE__SQLITE_PATH"] = str(db)
    os.environ["MEMTOMEM_INDEXING__MEMORY_DIRS"] = json.dumps([str(kr_notes)])
    os.environ["MEMTOMEM_EMBEDDING__PROVIDER"] = "ollama"
    os.environ["MEMTOMEM_EMBEDDING__MODEL"] = "nomic-embed-text"
    os.environ["MEMTOMEM_EMBEDDING__DIMENSION"] = "768"

    cfg = Mem2MemConfig()
    cfg.storage.sqlite_path = db
    cfg.indexing.memory_dirs = [kr_notes]
    cfg.search.tokenizer = tokenizer_name

    _cfg.load_config_overrides = lambda c: None
    try:
        local = await create_components(cfg)
    finally:
        _cfg.load_config_overrides = _orig_loader

    await local.index_engine.index_path(kr_notes)
    results, stats = await local.search_pipeline.search(query=query, top_k=5)

    print(f"--- {tokenizer_name} — {stats.final_total} hit(s) ---")
    for r in results:
        name = Path(r.chunk.metadata.source_file).name
        snippet = r.chunk.content[:60].replace("\n", " ")
        print(f"  [{r.rank}] {name:12s} score={r.score:.3f} — {snippet}...")
    if not results:
        print("  (no hits)")
    print()

    await close_components(local)


try:
    import kiwipiepy  # noqa: F401
except ImportError as e:
    raise ImportError(
        "kiwipiepy is required for the Korean tokenizer. Install with:\n"
        '  uv pip install "memtomem[korean]"'
    ) from e

await search_with_tokenizer("unicode61")
await search_with_tokenizer("kiwipiepy")

[04/11/26 11:05:37] INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=711310;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=267912;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=423507;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=606715;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=425917;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=488710;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

--- unicode61 — 2 hit(s) ---
  [1] deploy.md    score=0.033 — 우리 팀은 프로덕션 롤아웃을 시작하기 전에 스테이징에서 통합 테스트를 거친다. 각 서비스의 헬스체크가 성공해...
  [2] oncall.md    score=0.016 — 런북은 항상 최신 상태를 유지해야 한다. 지연 시간 경보가 울리면 Grafana 대시보드부터 열고, 오류율이...



                    INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=651168;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=52016;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

Quantization is not supported for ArchType::neon. Fall back to non-quantized model.
Quantization is not supported for ArchType::neon. Fall back to non-quantized model.


                    INFO     kiwipiepy tokenizer loaded successfully                            ]8;id=589954;file:///Users/pdstudio/Work/agent-harness/memtomem/packages/memtomem/src/memtomem/storage/fts_tokenizer.py\fts_tokenizer.py]8;;\:]8;id=384420;file:///Users/pdstudio/Work/agent-harness/memtomem/packages/memtomem/src/memtomem/storage/fts_tokenizer.py#49\49]8;;\

[04/11/26 11:05:38] INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=307259;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=700617;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=238822;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=165445;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

--- kiwipiepy — 2 hit(s) ---
  [1] deploy.md    score=0.033 — 우리 팀은 프로덕션 롤아웃을 시작하기 전에 스테이징에서 통합 테스트를 거친다. 각 서비스의 헬스체크가 성공해...
  [2] oncall.md    score=0.016 — 런북은 항상 최신 상태를 유지해야 한다. 지연 시간 경보가 울리면 Grafana 대시보드부터 열고, 오류율이...



### Reading the comparison

Both tokenizers return the same document here because the example is
small and unicode61's prefix wildcard compensates for its coarse
splitting. The takeaway is not that `kiwipiepy` wins in every case — it
is that `kiwipiepy` splits Korean into morphologically-correct tokens
before FTS5 sees them, which matters on larger corpora where vocabulary
is diverse and prefix matching gets noisy.

If you want to use `kiwipiepy` in a real memtomem install, either set
`MEMTOMEM_SEARCH__TOKENIZER=kiwipiepy` or rerun `mm init` and pick the
Korean option from the wizard.

## Cleanup

In [11]:
kr_tmp.cleanup()
tmp.cleanup()

for key in (
    "MEMTOMEM_STORAGE__SQLITE_PATH",
    "MEMTOMEM_INDEXING__MEMORY_DIRS",
    "MEMTOMEM_EMBEDDING__PROVIDER",
    "MEMTOMEM_EMBEDDING__MODEL",
    "MEMTOMEM_EMBEDDING__DIMENSION",
):
    os.environ.pop(key, None)

print("clean.")

clean.
